In [87]:
import pandas as pd
import math
chunks_data1 = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
first_chunk_data1 = next(chunks_data1)

chunks_data2 = pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000)
first_chunk_data2 = next(chunks_data2)
print(len(first_chunk_data2))

first_chunk_data1.head()


100000


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A10000012B7CGYKOMPQ4L,000100039X,Adam,"[0, 0]",Spiritually and mentally inspiring! A book tha...,5,Wonderful!,1355616000,"12 16, 2012"
1,A2S166WSCFIFP5,000100039X,"adead_poet@hotmail.com ""adead_poet@hotmail.com""","[0, 2]",This is one my must have books. It is a master...,5,close to god,1071100800,"12 11, 2003"
2,A1BM81XB4QHOA3,000100039X,"Ahoro Blethends ""Seriously""","[0, 0]",This book provides a reflection that you can a...,5,Must Read for Life Afficianados,1390003200,"01 18, 2014"
3,A1MOSTXNIO5MPJ,000100039X,Alan Krug,"[0, 0]",I first read THE PROPHET in college back in th...,5,Timeless for every good and bad time in your l...,1317081600,"09 27, 2011"
4,A2XQ5LZHTD4AFT,000100039X,Alaturka,"[7, 9]",A timeless classic. It is a very demanding an...,5,A Modern Rumi,1033948800,"10 7, 2002"


In [88]:
df_data1 = first_chunk_data1
df_data1["user_id"] = pd.factorize(df_data1["reviewerID"])[0]
df_data1["item_id"] = pd.factorize(df_data1["asin"])[0]
df_data1["interaction"] = 1


df_data2 = first_chunk_data2
df_data2["user_id"] = pd.factorize(df_data2["reviewerID"])[0]
df_data2["item_id"] = pd.factorize(df_data2["asin"])[0]
df_data2["interaction"] = 1



In [89]:
df_data1 = df_data1.sort_values(["user_id", "unixReviewTime"]).copy()

df_test_data1 = df_data1.groupby("user_id").tail(1).copy()
df_train_data1 = df_data1.drop(df_test_data1.index).copy()

df_train_data1 = df_train_data1.reset_index(drop=True)
df_test_data1 = df_test_data1.reset_index(drop=True)



df_data2 = df_data2.sort_values(["user_id", "unixReviewTime"]).copy()

df_test_data2 = df_data2.groupby("user_id").tail(1).copy()
df_train_data2 = df_data2.drop(df_test_data2.index).copy()

df_train_data2 = df_train_data2.reset_index(drop=True)
df_test_data2 = df_test_data2.reset_index(drop=True)


# MostPopular section

In [90]:
item_popularity = (
    df_train_data1.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()
user_seen_train = df_train_data1.groupby("user_id")["item_id"].apply(set).to_dict()

print(popular_items)

[591, 11, 552, 1552, 244, 601, 42, 1873, 1684, 2040, 172, 321, 367, 1310, 315, 758, 814, 391, 2607, 2447, 817, 2135, 195, 1693, 2146, 280, 13, 2351, 593, 691, 757, 292, 2328, 260, 2005, 2011, 274, 386, 2403, 1779, 190, 373, 320, 2038, 2587, 560, 2421, 2169, 2603, 1871, 2103, 457, 136, 1152, 2151, 2520, 1863, 1459, 2393, 1461, 587, 521, 39, 0, 1472, 1460, 2064, 38, 293, 2614, 1643, 1833, 1838, 437, 2375, 1416, 441, 2525, 2524, 1228, 477, 1981, 1205, 1327, 1368, 2523, 276, 2439, 277, 2522, 1158, 159, 1692, 1988, 2506, 509, 1995, 142, 2440, 180, 1581, 249, 819, 2157, 2254, 400, 165, 1058, 2012, 2554, 1178, 2325, 1900, 1332, 574, 833, 219, 2108, 1359, 294, 364, 575, 1296, 325, 2615, 1880, 2019, 415, 2576, 878, 1153, 2102, 553, 2359, 455, 1093, 1477, 446, 1443, 1832, 1361, 635, 1525, 2058, 745, 1336, 2118, 1805, 1388, 2110, 2109, 2162, 1331, 1257, 569, 241, 160, 1656, 576, 2507, 2178, 2153, 409, 2635, 69, 716, 1269, 1284, 1341, 1735, 1734, 317, 2424, 1865, 1732, 2211, 2390, 1276, 436, 2427,

In [91]:
def recommend_most_popular(user_id, popular_items, user_seen_train, k=10):
    seen = user_seen_train.get(user_id, set())
    recs = []
    for item in popular_items:
        if item not in seen:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item,k):
    return 1.0/k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0


def map(predictions, ground_truth, k=None):
    total_ap = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        hits = 0
        sum_precisions = 0

        for i, item in enumerate(ranked, start=1):
            if item in relevant:
                hits += 1
                sum_precisions += hits / i

        total_ap += sum_precisions / len(relevant)
        users += 1

    return total_ap / users if users > 0 else 0


def mrr(predictions, ground_truth, k=None):
    total = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        for rank, item in enumerate(ranked, start=1):
            if item in relevant:
                total += 1 / rank
                break

        users += 1

    return total / users if users > 0 else 0

In [92]:
K = 10
recalls = []
ndcgs = []
precisions = []
ground_truths = {}
all_recs = {}

for _, row in df_test_data1.iterrows():
    user_id = row["user_id"]
    ground_truth_item = row["item_id"]

    recs = recommend_most_popular(user_id, popular_items, user_seen_train, k=K)
    ground_truths[user_id] = [ground_truth_item]
    all_recs[user_id] = recs
    recalls.append(recall_at_k(recs, ground_truth_item))
    precisions.append(precision_at_k(recs,ground_truth_item,K))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

# PROBABLY NEED TO RE-WRITE THOSE FOR THE REAL THING COZ I REFUSE TO BELIEVE EACH USER WILL ACTUALLY ONLY HAVE ONE GROUND TRUTH ITEM
print(f"Users evaluated: {len(recalls)}")
# metrics to use for ranking specifically: Precision@K and Recall@K
print(f"Precision@{K}: {sum(precisions)/len(precisions):.4f}")
print(f"Recall@{K}: {sum(recalls)/len(recalls):.4f}")
# also nice for within-list ordering checks: NDCG@K, MAP AND MRR
print(f"NDCG@{K}: {sum(ndcgs)/len(ndcgs):.4f}")
print("MAP:", map(all_recs,ground_truths,K))
print("MRR:", mrr(all_recs,ground_truths,K))

Users evaluated: 68136
Precision@10: 0.0200
Recall@10: 0.2004
NDCG@10: 0.1069
MAP: 0.07814970549098915
MRR: 0.07814970549098915


## BPR



In [ ]:

import math
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking
from scipy.stats import wilcoxon
# =========================================================
# 1) Load both domains (for fair comparison with LightGCN/Joint BPR)
# =========================================================
books_chunk = next(pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000))
movies_chunk = next(pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000))
global_single_precision = []
global_single_recall = []
global_single_ndcg = []
global_single_precisions = []
global_single_recalls = []
global_single_ndcgs = []
books = books_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()
movies = movies_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()

print("Books interactions:", len(books))
print("Movies interactions:", len(movies))

# =========================================================
# 2) Keep only SHARED users (matching LightGCN/Joint BPR filtering)
# =========================================================
shared_users = set(books["reviewerID"]).intersection(set(movies["reviewerID"]))

books = books[books["reviewerID"].isin(shared_users)].copy()
movies = movies[movies["reviewerID"].isin(shared_users)].copy()

print("Shared users:", len(shared_users))
print("Books interactions after filter:", len(books))
print("Movies interactions after filter:", len(movies))

# =========================================================
# 3) Convert to joint format with domain prefixes
# =========================================================
def to_bpr_format(df, domain_name):
    out = pd.DataFrame({
        "user": df["reviewerID"].astype(str),
        "item": domain_name + "::" + df["asin"].astype(str),
        "interaction": 1.0,
        "time": df["unixReviewTime"].astype(np.int64),
        "domain": domain_name
    })
    return out

books_df = to_bpr_format(books, "Books")
movies_df = to_bpr_format(movies, "Movies")

SOURCE_NAME = "Books"
TARGET_NAME = "Movies"

print(f"\n{SOURCE_NAME} interactions:", len(books_df))
print(f"{TARGET_NAME} interactions:", len(movies_df))

# =========================================================
# 4) Helper functions for cold-start splitting
# =========================================================
def filter_users_for_cold_start(source_df, target_df, cold_k):
    """Filter users who have enough interactions in both domains"""
    source_counts = source_df["user"].value_counts()
    target_counts = target_df["user"].value_counts()
    
    valid_users = set(source_counts[source_counts >= 1].index) & \
                  set(target_counts[target_counts >= cold_k + 1].index)
    
    source_f = source_df[source_df["user"].isin(valid_users)].copy()
    target_f = target_df[target_df["user"].isin(valid_users)].copy()
    
    return source_f, target_f, valid_users

def make_target_cold_start_split(target_df, cold_k):
    """Keep only first cold_k interactions per user in target domain"""
    target_df = target_df.sort_values(["user", "time"]).copy()
    
    test_df = target_df.groupby("user").tail(1).copy()
    remaining = target_df.drop(test_df.index).copy()
    
    train_df = remaining.groupby("user", group_keys=False).head(cold_k).copy()
    
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

# =========================================================
# 5) Metric functions
# =========================================================
def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item, k):
    return 1.0 / k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0


# =========================================================
# 6) Train and evaluate BPR (single-domain and cross-domain)
# =========================================================
def train_bpr_from_df(train_df, factors=64, iterations=100):
    """Train BPR model on given dataframe"""
    train_df = train_df.copy()
    
    user_ids = sorted(train_df["user"].unique())
    item_ids = sorted(train_df["item"].unique())
    
    user2id = {old: new for new, old in enumerate(user_ids)}
    item2id = {old: new for new, old in enumerate(item_ids)}
    id2item = {new: old for old, new in item2id.items()}
    
    train_df["user_id"] = train_df["user"].map(user2id).astype(int)
    train_df["item_id"] = train_df["item"].map(item2id).astype(int)
    
    user_items_csr = csr_matrix(
        (
            train_df["interaction"].astype(np.float32),
            (train_df["user_id"], train_df["item_id"])
        ),
        shape=(len(user2id), len(item2id))
    )
    
    model = BayesianPersonalizedRanking(
        factors=factors,
        learning_rate=0.05,
        regularization=0.01,
        iterations=iterations,
        random_state=42
    )
    model.fit(user_items_csr, show_progress=False)
    
    return model, user_items_csr, user2id, item2id, id2item

def evaluate_bpr_target_only(model, user_items_csr, user2id, item2id, id2item, test_df, k=10, target_prefix="Movies::"):
    """Evaluate BPR, filtering recommendations to target domain only"""
    recalls = []
    precisions = []
    ndcgs = []
    all_recs = {}
    ground_truths = {}
    
    max_user_index = user_items_csr.shape[0] - 1
    
    for _, row in test_df.iterrows():
        user = row["user"]
        gt_item = row["item"]
        
        if user not in user2id or gt_item not in item2id:
            continue
        
        user_id = user2id[user]
        if user_id < 0 or user_id > max_user_index:
            continue
        
        # Ask for more candidates since we'll filter
        recs, scores = model.recommend(
            userid=user_id,
            user_items=user_items_csr[user_id],
            N=200,
            filter_already_liked_items=True
        )
        
        rec_items = [id2item[int(i)] for i in recs.tolist()]
        rec_items = [item for item in rec_items if item.startswith(target_prefix)][:k]
        
        all_recs[user] = rec_items
        ground_truths[user] = [gt_item]
        
        recalls.append(recall_at_k(rec_items, gt_item))
        precisions.append(precision_at_k(rec_items, gt_item, k))
        ndcgs.append(ndcg_at_k(rec_items, gt_item))
        global_single_recall.append(recall_at_k(rec_items, gt_item))
        global_single_precision.append(precision_at_k(rec_items, gt_item, k))
        global_single_ndcg.append(ndcg_at_k(rec_items, gt_item))
    
    return {
        "users_evaluated": len(recalls),
        f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
        f"MAP@{k}": mean_average_precision(all_recs, ground_truths, k),
        f"MRR@{k}": mean_reciprocal_rank(all_recs, ground_truths, k),
    }

def filter_test_seen_in_train(train_df, test_df):
    """Remove test items unseen in training"""
    train_users = set(train_df["user"].unique())
    train_items = set(train_df["item"].unique())
    
    test_df = test_df[
        test_df["user"].isin(train_users) &
        test_df["item"].isin(train_items)
    ].copy()
    
    return test_df.reset_index(drop=True)

# =========================================================
# 7) Run BPR experiments (single-domain vs cross-domain)
# =========================================================
def run_bpr_experiment(source_df, target_df, cold_k, k_eval=10):
    """Run both single-domain and cross-domain BPR experiments"""
    
    print(f"\n========== BPR: Cold-start level = keep {cold_k} target training interactions/user ==========")
    
    # Filter users for cold-start feasibility
    source_f, target_f, valid_users = filter_users_for_cold_start(source_df, target_df, cold_k)
    target_train, target_test = make_target_cold_start_split(target_f, cold_k)
    
    # Single-domain train: target only
    train_single = target_train[["user", "item", "interaction", "time"]].copy()
    test_single = filter_test_seen_in_train(train_single, target_test)
    
    # Cross-domain train: source + target
    train_cross = pd.concat([
        source_f[["user", "item", "interaction", "time"]],
        target_train[["user", "item", "interaction", "time"]]
    ], ignore_index=True)
    test_cross = filter_test_seen_in_train(train_cross, target_test)
    
    print(f"Single-domain train: {len(train_single)}, test: {len(test_single)}")
    print(f"Cross-domain train: {len(train_cross)}, test: {len(test_cross)}")
    
    # Train and evaluate single-domain
    model_single, user_items_csr_single, user2id_s, item2id_s, id2item_s = train_bpr_from_df(train_single)
    result_single = evaluate_bpr_target_only(
        model_single, user_items_csr_single, user2id_s, item2id_s, id2item_s,
        test_single, k=k_eval, target_prefix=f"{TARGET_NAME}::"
    )
    
    # Train and evaluate cross-domain
    model_cross, user_items_csr_cross, user2id_c, item2id_c, id2item_c = train_bpr_from_df(train_cross)
    result_cross = evaluate_bpr_target_only(
        model_cross, user_items_csr_cross, user2id_c, item2id_c, id2item_c,
        test_cross, k=k_eval, target_prefix=f"{TARGET_NAME}::"
    )
    
    return result_single, result_cross

# =========================================================
# 8) Run multiple cold-start severities
# =========================================================
cold_levels = [1, 2, 5]  # <-- CHANGE to control cold-start severity
bpr_results = []

for cold_k in cold_levels:
    single_res, cross_res = run_bpr_experiment(
        books_df, movies_df, cold_k, k_eval=10
    )
    global_single_precisions.append(global_single_precision)
    global_single_precision = []
    global_single_recalls.append(global_single_recall)
    global_single_recall = []
    global_single_ndcgs.append(global_single_ndcg)
    global_single_ndcg = []
    print(len(global_single_precisions[0]))
    
    row = {
        "cold_k": cold_k,
        "single_users": single_res["users_evaluated"],
        "single_Precision@10": single_res["Precision@10"],
        "single_Recall@10": single_res["Recall@10"],
        "single_NDCG@10": single_res["NDCG@10"],
    }
    bpr_results.append(row)

bpr_results_df = pd.DataFrame(bpr_results)
print("\n=== BPR Results (Single-Domain vs Cross-Domain) ===")
print(bpr_results_df.to_string())

Books interactions: 100000
Movies interactions: 100000
Shared users: 5008
Books interactions after filter: 13294
Movies interactions after filter: 24354

Books interactions: 13294
Movies interactions: 24354

========== BPR: Cold-start level = keep 1 target training interactions/user ==========
Single-domain train: 2856, test: 2529
Cross-domain train: 11800, test: 2529

========== BPR: Cold-start level = keep 2 target training interactions/user ==========
Single-domain train: 3834, test: 1766
Cross-domain train: 10868, test: 1766

========== BPR: Cold-start level = keep 5 target training interactions/user ==========
Single-domain train: 4645, test: 873
Cross-domain train: 8930, test: 873

=== BPR Results (Single-Domain vs Cross-Domain) ===
   cold_k  single_users  single_Precision@10  single_Recall@10  single_NDCG@10
0       1          2529             0.000712          0.007117        0.003174
1       2          1766             0.001586          0.015855        0.007729
2       5     

# Lights GCN

In [94]:
import math
import numpy as np
import pandas as pd

from libreco.data import DatasetPure
from libreco.algorithms import LightGCN
from libreco.algorithms import LightGCN


# =========================================================
# 1) Load one chunk per domain
# =========================================================
books_chunk = next(pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000))
movies_chunk = next(pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000))

books = books_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()
movies = movies_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()

books["domain"] = "Books"
movies["domain"] = "Movies"

print("Books interactions:", len(books))
print("Movies interactions:", len(movies))

Books interactions: 100000
Movies interactions: 100000


In [95]:
# =========================================================
# 2) Keep only shared users
# =========================================================
shared_users = set(books["reviewerID"]).intersection(set(movies["reviewerID"]))

books = books[books["reviewerID"].isin(shared_users)].copy()
movies = movies[movies["reviewerID"].isin(shared_users)].copy()

print("Shared users:", len(shared_users))
print("Books interactions after shared-user filter:", len(books))
print("Movies interactions after shared-user filter:", len(movies))

Shared users: 5008
Books interactions after shared-user filter: 13294
Movies interactions after shared-user filter: 24354


In [96]:
# =========================================================
# 3) Convert to LibRecommender format
#    user is shared across domains
#    item ids must be domain-prefixed so item spaces do not collide
# =========================================================
def to_libreco_format(df):
    out = pd.DataFrame({
        "user": df["reviewerID"].astype(str),
        "item": df["domain"].astype(str) + "::" + df["asin"].astype(str),
        "label": 1.0,
        "time": df["unixReviewTime"].astype(np.int64),
        "domain": df["domain"].astype(str)
    })
    return out

books_df = to_libreco_format(books)
movies_df = to_libreco_format(movies)

print(books_df.head())
print(movies_df.head())

              user               item  label        time domain
1   A2S166WSCFIFP5  Books::000100039X    1.0  1071100800  Books
7   A29TRDMK51GKZR  Books::000100039X    1.0  1383436800  Books
19   A27ZH1AQORJ1L  Books::000100039X    1.0  1066003200  Books
34  A1NPNGWBVD9AK3  Books::000100039X    1.0   961804800  Books
46   AWLFVCT9128JV  Books::000100039X    1.0  1136851200  Books
              user                item  label        time  domain
8    AWF2S3UNW9UA0  Movies::0005019281    1.0  1386201600  Movies
28   AVWWFK3FZDEL2  Movies::0005019281    1.0  1059004800  Movies
64  A1ZIBVOIPBWR3U  Movies::0005019281    1.0  1387497600  Movies
66  A2PSMIRDHYYONP  Movies::0005019281    1.0  1386547200  Movies
71  A17TPT3FWAE5T1  Movies::0005019281    1.0  1008028800  Movies


In [97]:
# =========================================================
# 4) Choose source and target domains
#    Example:
#      source = Books
#      target = Movies
# =========================================================
source_df = books_df.copy()
target_df = movies_df.copy()

SOURCE_NAME = "Books"
TARGET_NAME = "Movies"

In [98]:
# =========================================================
# 5) Filter users so the experiment is feasible
#    Need:
#      - at least 1 source interaction
#      - enough target interactions to do train/test
#
#    For cold-start severity c:
#      keep c target interactions in train
#      last target interaction is held out for test
#
#    So each user needs at least c + 1 target interactions
# =========================================================
def filter_users_for_cold_start(source_df, target_df, cold_k):
    source_counts = source_df["user"].value_counts()
    target_counts = target_df["user"].value_counts()

    valid_users = set(source_counts[source_counts >= 1].index) & \
                  set(target_counts[target_counts >= cold_k + 1].index)

    source_f = source_df[source_df["user"].isin(valid_users)].copy()
    target_f = target_df[target_df["user"].isin(valid_users)].copy()

    return source_f, target_f, valid_users

In [99]:
# =========================================================
# 6) Build target cold-start split
#    For each user in target domain:
#      - last interaction -> test
#      - keep only first cold_k earlier interactions in target train
#
#    This simulates varying cold-start severity in target domain,
#    exactly like your report describes.
# =========================================================
def make_target_cold_start_split(target_df, cold_k):
    target_df = target_df.sort_values(["user", "time"]).copy()

    test_df = target_df.groupby("user").tail(1).copy()
    remaining = target_df.drop(test_df.index).copy()

    # keep earliest cold_k interactions from remaining as target-train
    target_train = (
        remaining.groupby("user", group_keys=False)
        .head(cold_k)
        .copy()
    )

    target_train = target_train.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    return target_train, test_df

In [100]:
# =========================================================
# 7) Build two train sets:
#    A) single-domain baseline = target train only
#    B) cross-domain model = source train + target train
# =========================================================
def build_experiment_frames(source_df, target_df, cold_k):
    source_f, target_f, valid_users = filter_users_for_cold_start(source_df, target_df, cold_k)
    target_train, target_test = make_target_cold_start_split(target_f, cold_k)

    # baseline: target only
    train_single = target_train[["user", "item", "label", "time"]].copy()

    # cross-domain: source + target
    train_cross = pd.concat([
        source_f[["user", "item", "label", "time"]],
        target_train[["user", "item", "label", "time"]]
    ], ignore_index=True)

    test_target = target_test[["user", "item", "label", "time"]].copy()

    return train_single, train_cross, test_target

In [101]:
# =========================================================
# 8) Remove test rows whose target item is unseen in train
#    Needed because the model cannot rank completely unseen items
# =========================================================
def filter_test_seen_in_train(train_df, test_df):
    train_users = set(train_df["user"].unique())
    train_items = set(train_df["item"].unique())

    test_df = test_df[
        test_df["user"].isin(train_users) &
        test_df["item"].isin(train_items)
    ].copy()

    return test_df.reset_index(drop=True)

In [102]:
# =========================================================
# 9) Metric functions
# =========================================================
def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item, k):
    return 1.0 / k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

def mean_average_precision(predictions, ground_truth, k=None):
    total_ap = 0.0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        hits = 0
        sum_precisions = 0.0

        for i, item in enumerate(ranked, start=1):
            if item in relevant:
                hits += 1
                sum_precisions += hits / i

        total_ap += sum_precisions / len(relevant)
        users += 1

    return total_ap / users if users > 0 else 0.0

def mean_reciprocal_rank(predictions, ground_truth, k=None):
    total = 0.0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        for rank, item in enumerate(ranked, start=1):
            if item in relevant:
                total += 1.0 / rank
                break

        users += 1

    return total / users if users > 0 else 0.0

In [103]:
# =========================================================
# 10) Train a LibRecommender LightGCN model
# =========================================================
def train_lightgcn_libreco(train_df, n_epochs=20, embed_size=64):
    train_data, data_info = DatasetPure.build_trainset(train_df)

    model = LightGCN(
        task="ranking",
        data_info=data_info,
        loss_type="bpr",
        embed_size=embed_size,
        n_epochs=n_epochs,
        lr=1e-3,
        batch_size=2048,
        num_neg=1,
        device="cuda",
        seed=42,
    )

    model.fit(
        train_data,
        neg_sampling=True,
        verbose=2,
        shuffle=True
    )

    return model, data_info

In [104]:
# =========================================================
# 11) Evaluate on target-domain held-out items
#     Also filter recommendations to TARGET domain only
# =========================================================
def evaluate_target_only(model, test_df, k=10, target_prefix="Movies::"):
    recalls = []
    precisions = []
    ndcgs = []
    all_recs = {}
    ground_truths = {}
    rows = []

    for _, row in test_df.iterrows():
        user = row["user"]
        gt_item = row["item"]

        # Ask for more than k because returned recs may include source-domain items
        rec_dict = model.recommend_user(
            user=user,
            n_rec=100,
            cold_start="popular",
            inner_id=False,
            filter_consumed=True,
            random_rec=False,
        )

        recs = rec_dict[user]
        if isinstance(recs, np.ndarray):
            recs = recs.tolist()
        else:
            recs = list(recs)

        # keep only target-domain items
        recs = [item for item in recs if item.startswith(target_prefix)][:k]

        all_recs[user] = recs
        ground_truths[user] = [gt_item]

        precision = precision_at_k(recs, gt_item, k)
        recall = recall_at_k(recs, gt_item)
        ndcg = ndcg_at_k(recs, gt_item)

        rows.append({
            "user": user,
            "gt_item": gt_item,
            "precision": precision,
            "recall": recall,
            "ndcg": ndcg,
            "recommendations": recs,
        })

        recalls.append(recall)
        precisions.append(precision)
        ndcgs.append(ndcg)

    per_user_df = pd.DataFrame(rows)

    return {
        "users_evaluated": len(recalls),
        f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
        f"MAP@{k}": mean_average_precision(all_recs, ground_truths, k),
        f"MRR@{k}": mean_reciprocal_rank(all_recs, ground_truths, k),
        "per_user_df": per_user_df,
    }

In [105]:
# =========================================================
# 12) Run one experiment for a given cold-start severity
# =========================================================
def run_one_experiment(source_df, target_df, cold_k, k_eval=10, n_epochs=20):
    print(f"\n========== Cold-start level: keep {cold_k} target train interactions/user ==========")

    train_single, train_cross, test_target = build_experiment_frames(source_df, target_df, cold_k)

    test_single = filter_test_seen_in_train(train_single, test_target)
    test_cross = filter_test_seen_in_train(train_cross, test_target)

    print("Single-domain train size:", len(train_single))
    print("Cross-domain train size:", len(train_cross))
    print("Single-domain test size:", len(test_single))
    print("Cross-domain test size:", len(test_cross))

    model_single, _ = train_lightgcn_libreco(train_single, n_epochs=n_epochs)
    result_single = evaluate_target_only(
        model_single,
        test_single,
        k=k_eval,
        target_prefix=f"{TARGET_NAME}::"
    )

    model_cross, _ = train_lightgcn_libreco(train_cross, n_epochs=n_epochs)
    result_cross = evaluate_target_only(
        model_cross,
        test_cross,
        k=k_eval,
        target_prefix=f"{TARGET_NAME}::"
    )

    return result_single, result_cross

In [106]:
# =========================================================
# 13) Run multiple cold-start severities
#     Example severities: 1, 2, 5
# =========================================================
cold_levels = [1, 2, 5]
results = []

for cold_k in cold_levels:
    single_res, cross_res = run_one_experiment(
        source_df=source_df,
        target_df=target_df,
        cold_k=cold_k,
        k_eval=10,
        n_epochs=20
    )

    stat, p_precision = wilcoxon(single_res["per_user_df"]["precision"], cross_res["per_user_df"]["precision"])
    stat, p_recall = wilcoxon(single_res["per_user_df"]["recall"], cross_res["per_user_df"]["recall"])
    stat, p_ndcg = wilcoxon(single_res["per_user_df"]["ndcg"], cross_res["per_user_df"]["ndcg"])
    row = {
        "cold_k": cold_k,
        "single_users": single_res["users_evaluated"],
        "cross_users": cross_res["users_evaluated"],
        "single_Precision@10": single_res["Precision@10"],
        "cross_Precision@10": cross_res["Precision@10"],
        "single_Recall@10": single_res["Recall@10"],
        "cross_Recall@10": cross_res["Recall@10"],
        "single_NDCG@10": single_res["NDCG@10"],
        "cross_NDCG@10": cross_res["NDCG@10"],
        "p_precision": p_precision,
        "p_recall": p_recall,
        "p_ndcg": p_ndcg
    }
    results.append(row)

results_df = pd.DataFrame(results)
print("\n=== Final Results ===")
print(results_df)


========== Cold-start level: keep 1 target train interactions/user ==========
Single-domain train size: 2856
Cross-domain train size: 11800
Single-domain test size: 2529
Cross-domain test size: 2529
Training start time: 2026-04-04 14:13:42


train: 100%|██████████| 2/2 [00:00<00:00, 24.14it/s]


Epoch 1 elapsed: 0.090s
	 train_loss: 0.6193


train: 100%|██████████| 2/2 [00:00<00:00, 100.63it/s]


Epoch 2 elapsed: 0.025s
	 train_loss: 0.6155


...

Epoch 18 elapsed: 0.084s
	 train_loss: 0.6323


train: 100%|██████████| 5/5 [00:00<00:00, 67.46it/s]


Epoch 19 elapsed: 0.077s
	 train_loss: 0.6259


train: 100%|██████████| 5/5 [00:00<00:00, 75.56it/s]


Epoch 20 elapsed: 0.072s
	 train_loss: 0.6188

=== Final Results ===
   cold_k  single_users  cross_users  single_Precision@10  cross_Precision@10  \
0       1          2529         2529             0.002887            0.007394   
1       2          1766         1766             0.002718            0.006965   
2       5           873          873             0.002520            0.005956   

   single_Recall@10  cross_Recall@10  single_NDCG@10  cross_NDCG@10  \
0          0.028865         0.073942        0.014160       0.040135   
1          0.027180         0.069649        0.014215       0.034188   
2          0.025200         0.059565        0.010830       0.028596   

    p_precision      p_recall        p_ndcg  
0  1.991449e-14  1.991449e-14  2.262302e-13  
1  8.034758e-10  8.034758e-10  1.374547e-06  
2  1.389647e-04  1.389647e-04  1.825205e-04  


In [107]:
print(train_lightgcn_libreco)

<function train_lightgcn_libreco at 0x000002113EFBA950>


## Joint BPR

In [109]:
import math
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

# =========================================================
# 1) Load one chunk per domain
# =========================================================
books_chunk = next(pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000))
movies_chunk = next(pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000))

books = books_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()
movies = movies_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()

books["domain"] = "Books"
movies["domain"] = "Movies"

print("Books interactions:", len(books))
print("Movies interactions:", len(movies))

# =========================================================
# 2) Keep only shared users
# =========================================================
shared_users = set(books["reviewerID"]).intersection(set(movies["reviewerID"]))

books = books[books["reviewerID"].isin(shared_users)].copy()
movies = movies[movies["reviewerID"].isin(shared_users)].copy()

print("Shared users:", len(shared_users))
print("Books interactions after shared-user filter:", len(books))
print("Movies interactions after shared-user filter:", len(movies))


# =========================================================
# 3) Convert to joint format
#    user ids are shared across domains
#    item ids are domain-prefixed so they do not collide
# =========================================================
def to_joint_format(df):
  out = pd.DataFrame({
    "user": df["reviewerID"].astype(str),
    "item": df["domain"].astype(str) + "::" + df["asin"].astype(str),
    "interaction": 1.0,
    "time": df["unixReviewTime"].astype(np.int64),
    "domain": df["domain"].astype(str)
  })
  return out


books_df = to_joint_format(books)
movies_df = to_joint_format(movies)

print(books_df.head())
print(movies_df.head())

# =========================================================
# 4) Choose source and target domains
# =========================================================
source_df = books_df.copy()
target_df = movies_df.copy()

SOURCE_NAME = "Books"
TARGET_NAME = "Movies"


# =========================================================
# 5) Filter users for cold-start setup
#    Need:
#      - at least 1 source interaction
#      - at least cold_k + 1 target interactions
# =========================================================
def filter_users_for_cold_start(source_df, target_df, cold_k):
  source_counts = source_df["user"].value_counts()
  target_counts = target_df["user"].value_counts()

  valid_users = set(source_counts[source_counts >= 1].index) & \
                set(target_counts[target_counts >= cold_k + 1].index)

  source_f = source_df[source_df["user"].isin(valid_users)].copy()
  target_f = target_df[target_df["user"].isin(valid_users)].copy()

  return source_f, target_f, valid_users


# =========================================================
# 6) Build target cold-start split
#    - last target interaction -> test
#    - first cold_k earlier interactions -> target train
# =========================================================
def make_target_cold_start_split(target_df, cold_k):
  target_df = target_df.sort_values(["user", "time"]).copy()

  test_df = target_df.groupby("user").tail(1).copy()
  remaining = target_df.drop(test_df.index).copy()

  target_train = (
    remaining.groupby("user", group_keys=False)
    .head(cold_k)
    .copy()
  )

  target_train = target_train.reset_index(drop=True)
  test_df = test_df.reset_index(drop=True)

  return target_train, test_df


# =========================================================
# 7) Build joint train/test
#    joint train = all source interactions + limited target train
# =========================================================
def build_joint_bpr_frames(source_df, target_df, cold_k):
  source_f, target_f, valid_users = filter_users_for_cold_start(source_df, target_df, cold_k)
  target_train, target_test = make_target_cold_start_split(target_f, cold_k)

  train_joint = pd.concat([
    source_f[["user", "item", "interaction", "time"]],
    target_train[["user", "item", "interaction", "time"]]
  ], ignore_index=True)

  test_target = target_test[["user", "item", "interaction", "time"]].copy()

  return train_joint, test_target


# =========================================================
# 8) Remove test rows whose user/item is unseen in train
# =========================================================
def filter_test_seen_in_train(train_df, test_df):
  train_users = set(train_df["user"].unique())
  train_items = set(train_df["item"].unique())

  test_df = test_df[
    test_df["user"].isin(train_users) &
    test_df["item"].isin(train_items)
    ].copy()

  return test_df.reset_index(drop=True)


# =========================================================
# 9) Metrics
# =========================================================
def recall_at_k(recommended_items, ground_truth_item):
  return 1.0 if ground_truth_item in recommended_items else 0.0


def precision_at_k(recommended_items, ground_truth_item, k):
  return 1.0 / k if ground_truth_item in recommended_items else 0.0


def ndcg_at_k(recommended_items, ground_truth_item):
  if ground_truth_item in recommended_items:
    rank = recommended_items.index(ground_truth_item) + 1
    return 1.0 / math.log2(rank + 1)
  return 0.0


def mean_average_precision(predictions, ground_truth, k=None):
  total_ap = 0.0
  users = 0

  for user in predictions:
    if user not in ground_truth or len(ground_truth[user]) == 0:
      continue

    ranked = predictions[user][:k] if k else predictions[user]
    relevant = ground_truth[user]

    hits = 0
    sum_precisions = 0.0

    for i, item in enumerate(ranked, start=1):
      if item in relevant:
        hits += 1
        sum_precisions += hits / i

    total_ap += sum_precisions / len(relevant)
    users += 1

  return total_ap / users if users > 0 else 0.0


def mean_reciprocal_rank(predictions, ground_truth, k=None):
  total = 0.0
  users = 0

  for user in predictions:
    if user not in ground_truth or len(ground_truth[user]) == 0:
      continue

    ranked = predictions[user][:k] if k else predictions[user]
    relevant = ground_truth[user]

    for rank, item in enumerate(ranked, start=1):
      if item in relevant:
        total += 1.0 / rank
        break

    users += 1

  return total / users if users > 0 else 0.0


# =========================================================
# 10) Build sparse matrix with tight remapping
# =========================================================
def build_joint_sparse_matrix(train_df):
  train_df = train_df.copy()

  user_ids = sorted(train_df["user"].unique())
  item_ids = sorted(train_df["item"].unique())

  user2id = {u: i for i, u in enumerate(user_ids)}
  item2id = {it: i for i, it in enumerate(item_ids)}
  id2item = {i: it for it, i in item2id.items()}

  train_df["user_id"] = train_df["user"].map(user2id).astype(int)
  train_df["item_id"] = train_df["item"].map(item2id).astype(int)

  num_users = int(train_df["user_id"].max()) + 1
  num_items = int(train_df["item_id"].max()) + 1

  print("Users:", num_users, "Items:", num_items)

  user_items_csr = csr_matrix(
    (
      train_df["interaction"].astype(np.float32),
      (train_df["user_id"], train_df["item_id"])
    ),
    shape=(num_users, num_items)
  )

  return train_df, user_items_csr, user2id, item2id, id2item


# =========================================================
# 11) Train joint BPR
#    IMPORTANT:
#    implicit BPR should be fit on USER-ITEM matrix here
# =========================================================
def train_joint_bpr(train_df, factors=64, lr=0.05, reg=0.01, iterations=100):
  train_mapped, user_items_csr, user2id, item2id, id2item = build_joint_sparse_matrix(train_df)

  model = BayesianPersonalizedRanking(
    factors=factors,
    learning_rate=lr,
    regularization=reg,
    iterations=iterations,
    random_state=42
  )

  model.fit(user_items_csr, show_progress=True)

  print("Model user factors:", model.user_factors.shape)
  print("Model item factors:", model.item_factors.shape)

  return model, user_items_csr, user2id, item2id, id2item


# =========================================================
# 12) Evaluate only on target-domain items
#    same idea as your LightGCN evaluation:
#    recommend broadly, then filter to target prefix
# =========================================================
def evaluate_joint_bpr_target_only(model, user_items_csr, user2id, item2id, id2item, test_df, k=10,
                                   target_prefix="Movies::"):
  recalls = []
  precisions = []
  ndcgs = []
  all_recs = {}
  ground_truths = {}
  rows = []

  max_user_index = user_items_csr.shape[0] - 1
  num_items = user_items_csr.shape[1]

  for _, row in test_df.iterrows():
    user = row["user"]
    gt_item = row["item"]

    if user not in user2id:
      continue
    if gt_item not in item2id:
      continue

    user_id = int(user2id[user])

    if user_id < 0 or user_id > max_user_index:
      continue

    # ask for more than k because many recs may belong to the source domain
    n_candidates = min(200, num_items)

    rec_ids, scores = model.recommend(
      userid=user_id,
      user_items=user_items_csr[user_id],
      N=n_candidates,
      filter_already_liked_items=True
    )

    if isinstance(rec_ids, np.ndarray):
      rec_ids = rec_ids.tolist()
    else:
      rec_ids = list(rec_ids)

    rec_items = [id2item[i] for i in rec_ids]
    rec_items = [item for item in rec_items if item.startswith(target_prefix)][:k]

    all_recs[user] = rec_items
    ground_truths[user] = [gt_item]

    rows.append({
            "user": user,
            "gt_item": gt_item,
            "precision": precision_at_k(rec_items, gt_item, k),
            "recall": recall_at_k(rec_items, gt_item),
            "ndcg": ndcg_at_k(rec_items, gt_item),
            "recommendations": recs,
        })

    per_user_df = pd.DataFrame(rows)

    recalls.append(recall_at_k(rec_items, gt_item))
    precisions.append(precision_at_k(rec_items, gt_item, k))
    ndcgs.append(ndcg_at_k(rec_items, gt_item))

  return {
    "users_evaluated": len(recalls),
    f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
    f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
    f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
    f"MAP@{k}": mean_average_precision(all_recs, ground_truths, k),
    f"MRR@{k}": mean_reciprocal_rank(all_recs, ground_truths, k),
    "per_user_df": per_user_df,
  }


# =========================================================
# 13) Run one experiment
# =========================================================
def run_joint_bpr_experiment(source_df, target_df, cold_k, k_eval=10, factors=64, iterations=100):
  print(f"\n========== Joint BPR cold-start level: keep {cold_k} target train interactions/user ==========")

  train_joint, test_target = build_joint_bpr_frames(source_df, target_df, cold_k)
  test_target = filter_test_seen_in_train(train_joint, test_target)

  print("Joint train size:", len(train_joint))
  print("Joint test size:", len(test_target))

  model, user_items_csr, user2id, item2id, id2item = train_joint_bpr(
    train_joint,
    factors=factors,
    lr=0.05,
    reg=0.01,
    iterations=iterations
  )

  valid_users = set(user2id.keys())
  valid_items = set(item2id.keys())

  test_target = test_target[
    test_target["user"].isin(valid_users) &
    test_target["item"].isin(valid_items)
    ].copy()

  results = evaluate_joint_bpr_target_only(
    model=model,
    user_items_csr=user_items_csr,
    user2id=user2id,
    item2id=item2id,
    id2item=id2item,
    test_df=test_target,
    k=k_eval,
    target_prefix=f"{TARGET_NAME}::"
  )

  return results


# =========================================================
# 14) Run multiple cold-start severities
# =========================================================
cold_levels = [1, 2, 5]
results = []
temp_i = 0 
for cold_k in cold_levels:
  joint_res = run_joint_bpr_experiment(
    source_df=source_df,
    target_df=target_df,
    cold_k=cold_k,
    k_eval=10,
    factors=64,
    iterations=100
  )
  print(len(global_single_precision))
  print(len(joint_res["per_user_df"]["precision"]))
  stat, p_precision = wilcoxon(global_single_precisions[temp_i], joint_res["per_user_df"]["precision"])
  stat, p_recall = wilcoxon(global_single_recalls[temp_i], joint_res["per_user_df"]["recall"])
  stat, p_ndcg = wilcoxon(global_single_ndcgs[temp_i], joint_res["per_user_df"]["ndcg"])
  row = {
    "cold_k": cold_k,
    "joint_users": joint_res["users_evaluated"],
    "joint_Precision@10": joint_res["Precision@10"],
    "joint_Recall@10": joint_res["Recall@10"],
    "joint_NDCG@10": joint_res["NDCG@10"],
    "joint_MAP@10": joint_res["MAP@10"],
    "joint_MRR@10": joint_res["MRR@10"],
  }
  results.append(row)
  temp_i = temp_i + 1

results_df = pd.DataFrame(results)
print("\n=== Joint BPR-MF Results ===")
print(results_df)

Books interactions: 100000
Movies interactions: 100000
Shared users: 5008
Books interactions after shared-user filter: 13294
Movies interactions after shared-user filter: 24354
              user               item  interaction        time domain
1   A2S166WSCFIFP5  Books::000100039X          1.0  1071100800  Books
7   A29TRDMK51GKZR  Books::000100039X          1.0  1383436800  Books
19   A27ZH1AQORJ1L  Books::000100039X          1.0  1066003200  Books
34  A1NPNGWBVD9AK3  Books::000100039X          1.0   961804800  Books
46   AWLFVCT9128JV  Books::000100039X          1.0  1136851200  Books
              user                item  interaction        time  domain
8    AWF2S3UNW9UA0  Movies::0005019281          1.0  1386201600  Movies
28   AVWWFK3FZDEL2  Movies::0005019281          1.0  1059004800  Movies
64  A1ZIBVOIPBWR3U  Movies::0005019281          1.0  1387497600  Movies
66  A2PSMIRDHYYONP  Movies::0005019281          1.0  1386547200  Movies
71  A17TPT3FWAE5T1  Movies::0005019281     

100%|██████████| 100/100 [00:00<00:00, 384.37it/s, train_auc=99.95%, skipped=1.44%]


Model user factors: (2856, 65)
Model item factors: (2673, 65)
0
2529


ValueError: Array shapes are incompatible for broadcasting.